<a href="https://colab.research.google.com/github/Rosamii1/AI4SG-Team-3/blob/main/AI4SG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip install -q google-generativeai

In [27]:
import google.generativeai as genai
from google.colab import userdata
import json
import time

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
print("Gemini initialized successfully.")

Gemini initialized successfully.


In [28]:
from google.colab import files
from PIL import Image as PILImage

In [29]:
# # The schema — the form the AI must fill in exactly.
# schema = """
# Extract information from this input.
# Return ONLY valid JSON with exactly these five fields:
# {
#   "location": string (the city or location described),
#   "rent_amount": number (the dollar amount of rent),
#   "urgency": "LOW" or "MEDIUM" or "HIGH",
#   "plan": string (What is currently planned for the site?),
#   "resident_language": string (language the resident wrote in, e.g. "English", "Spanish", "Vietnamese")
# }
# Urgency guide: LOW = small litter or cosmetic issue, MEDIUM = large items or ongoing problem, HIGH = hazardous materials or immediate safety risk.
# No explanation. No markdown. JSON only.
# """

In [30]:
# def extract_structured(message):
#     m = genai.GenerativeModel(
#         model_name="gemini-2.5-flash",
#         system_instruction=system_instruction_text
#     )
#     response = m.generate_content(message)
#     time.sleep(12)  # stays under free tier rate limit
#     raw = response.text.strip()
#     # Strip markdown code fences if present
#     if raw.startswith("```"):
#         raw = raw.split("\n", 1)[1].rsplit("```", 1)[0].strip()
#     return json.loads(raw)

In [31]:
# def get_completion(prompt, max_retries=3):
#     m = genai.GenerativeModel(
#         model_name="gemini-2.5-flash",
#         system_instruction=system_instruction_text
#     )

#     retries = 0
#     while retries < max_retries:
#         try:
#             response = m.generate_content(prompt)
#             time.sleep(12)  # stays under free tier rate limit (5 requests/min)
#             return response.text
#         except TooManyRequests as e:
#             retries += 1
#             if retries == max_retries:
#                 raise e # Re-raise if max retries reached

#             # Extract retry time from the error message
#             match = re.search(r'Please retry in (\d+\.?\d*)s\.', str(e))
#             wait_time = 60 # Default wait time if not found
#             if match:
#                 wait_time = float(match.group(1))

#             print(f"Quota exceeded. Retrying in {wait_time:.2f} seconds... (Attempt {retries}/{max_retries})")
#             time.sleep(wait_time + 1) # Add a small buffer
#         except Exception as e:
#             # Handle other potential errors
#             raise e

#     # Should not reach here if max_retries is handled correctly
#     return ""


In [32]:
system_instruction_text = """
Find the location of these coordinates
"""

In [33]:
civic_questions = [
    ("FIND LOCATION", "What's the location of this plot? what property problems does this plot of land face?"),
    ("PROBLEM",  "Can you help me identify the problem, is it limited housing with spiking prices or something else?"),
    ("TRANSIT",  "Can you help me identify what type of transit is near this land"),
    ("IMPACT",   "What are the community impacts of the land owning?"),
    ("AREA TYPE", "What type of area is this image, is it suburban, urban, or downtown?"),
    ("ACTION",   "Which company or landowner should the government try to convey an act of plan.")
]

civic_results = {"answers": {}, "total_tokens": 0}

In [34]:
def analyze_image(image_path, question):
    """
    Send an image + a question to Gemini and return the response.
    Returns: (response_text, usage_metadata)
    """
    m = genai.GenerativeModel(model_name="gemini-2.5-flash")
    img = PILImage.open(image_path)
    response = m.generate_content([question, img])
    time.sleep(12)  # stays under free tier rate limit
    return response.text, response.usage_metadata

In [35]:
def analyze_text(message, question):
    m = genai.GenerativeModel(
        model_name="gemini-2.5-flash",
        system_instruction=system_instruction_text
    )
    response = m.generate_content([question, message])
    return response.text, response.usage_metadata

In [36]:
q = ""
while True:
  q = input("Are you submitting an image or text message? Press 1 for Text, press 2 for Image ")
  if q == "1":
    break
  elif q == "2":
    break
  else:
    print("Invalid input. Please try again.")

if q == "1":
  resident_message = input("Enter coordinates ")
  print("\n--- AI Response ---")

  for label, question in civic_questions:
      print(f"--- {label} ---")
      answer, usage = analyze_text(resident_message, question)
      civic_results["answers"][label] = answer
      civic_results["total_tokens"] += usage.total_token_count
      print(answer)
      print()
      import time
      # Increase sleep to ensure we stay under the free tier rate limit.
      # The error message suggests retrying, so a longer sleep is needed.
      # analyze_image already has a 12 second sleep, so we add 60 seconds here for a total of 72 seconds per call.
      time.sleep(60)

  print(f"--- Total tokens used: {civic_results['total_tokens']} ---")
  print("Running on Gemini free tier — no cost.")

elif q == "2":
  # Upload your image from your computer.
  uploaded = files.upload()

  # Get the filename of the uploaded file.
  image_filename = list(uploaded.keys())[0]

  # Display the uploaded image so you can see what the model will analyze.
  img = PILImage.open(image_filename)
  print(f"Uploaded: {image_filename}")
  print(f"Image size: {img.size[0]}x{img.size[1]} pixels")
  display(img)

  for label, question in civic_questions:
      print(f"--- {label} ---")
      answer, usage = analyze_image(image_filename, question)
      civic_results["answers"][label] = answer
      civic_results["total_tokens"] += usage.total_token_count
      print(answer)
      print()
      import time
      # Increase sleep to ensure we stay under the free tier rate limit.
      # The error message suggests retrying, so a longer sleep is needed.
      # analyze_image already has a 12 second sleep, so we add 60 seconds here for a total of 72 seconds per call.
      time.sleep(60)

  print(f"--- Total tokens used: {civic_results['total_tokens']} ---")
  print("Running on Gemini free tier — no cost.")

Are you submitting an image or text message? Press 1 for Text, press 2 for Image 1
Enter coordinates 37°20'03.2"N 121°52'36.6"W

--- AI Response ---
--- FIND LOCATION ---
The coordinates 37°20'03.2"N 121°52'36.6"W point to a specific location in **South San Jose, California, USA.**

More specifically, this plot of land is located on the **east side of Senter Road, just south of Monterey Road, in San Jose's Edenvale district.** It appears to be a large commercial or industrial property, possibly one of the Intel facilities or a similar tech/industrial complex in the area. It is situated within the heart of **Silicon Valley**.

Given its location in San Jose, California, this plot of land (or property built upon it) faces several common property problems and considerations:

### Property Problems & Considerations:

1.  **Earthquake Risk:** San Jose is located in a highly active seismic zone, with numerous fault lines nearby (including the San Andreas Fault system).
    *   **Problem:** S

KeyboardInterrupt: 